# Lab 4.05 - Bivariate Analysis of Qualitative Data


In [ ]:
# Importing the necessary packages
import numpy as np                                  # "Scientific computing"
import scipy.stats as stats                         # Statistical tests

import pandas as pd                                 # Data Frame
from pandas.api.types import CategoricalDtype

import matplotlib.pyplot as plt                     # Basic visualisation
from statsmodels.graphics.mosaicplot import mosaic  # Mosaic diagram
import seaborn as sns                               # Advanced data visualisation

## Exercise 5 - Survey of Australian Students

Load the data file data/survey.csv. It contains the result of a survey of students from an Australian university.

We want to investigate the relationship between some discrete (nominal or ordinal) variables in this dataset. For any pairs of variables listed below, follow these steps:

* First, think about what exactly you expect for the given combination of variables.
* Make a frequency table for the two variables. The (presumably) independent variable comes first.
* Plot a graph visualizing the relationship between the two variables.
- Looking at the chart, do you expect a rather high or rather low value for the $\chi^2$ statistic? Why?
* Run the $\chi^2$ test to determine whether there is a relationship between the two variables. Calculate the $\chi^2$ statistic, the critical limit $g$ and the $p$ value, each for significance level $\alpha = 0.05$.
* Should we accept or reject the null hypothesis? What exactly does that mean for the relationship between the two variables? In other words, formulate an answer to the research question.
* Calculate Cramér's V. Do you come to a similar conclusion as with the $\chi^2$ test?


The variables to be investigated:

| Independent variabele          | Dependent variabele                        |
|:------------------------------ |:-------------------------------------------|
| `Exer` (practicing sports)     | `Smoke`                                    |
| `Sex` (gender)                 | `Smoke`                                    |
| `W.Hnd` (dominant hand)        | `Fold` (top hand when you cross your arms) |
| `Sex`                          | `W.Hnd`                                    |

Results of the main calculations (rounded up to 3 decimal places):

- `Exer/Smoke`: χ² ≈ 5.489, g ≈ 12.592, p ≈ 0.483
- `W.Hnd/Fold`: χ² ≈ 1.581, g ≈ 5.992, p ≈ 0.454
- `Sex/Smoke`: χ² ≈ 3.554, g ≈ 7.815, p ≈ 0.314
- `Sex/W.Hnd`: χ² ≈ 0.236, g ≈ 3.842, p ≈ 0.627

Read the dataset. **Be careful!** The `read_csv()` function does not handle missing values in this dataset correctly. Missing values are represented by the string "NA", which is automatically detected by `read_csv()` as a missing value. However, it will also convert the value 'None' in the column 'Exer' to a missing value, which is not correct. To prevent this, set the parameters `na_values` to the *only* valid missing value "NA" and `keep_default_na` to `False` when calling `read_csv()`. This way, the value 'None' will be read as a valid value in the 'Exer' column.

What are the different values for Exer and Smoke?  
Change both variables to ordinal variables with a specific order.

In [ ]:
# Inlezen van de data met specifieke NA-instellingen
survey = pd.read_csv('data/survey.csv', na_values=['NA'], keep_default_na=False)

# Definiëren van de ordinale volgorde
exer_type = CategoricalDtype(categories=['None', 'Some', 'Freq'], ordered=True)
smoke_type = CategoricalDtype(categories=['Never', 'Occas', 'Regul', 'Heavy'], ordered=True)

# Typeconversie toepassen
survey['Exer'] = survey['Exer'].astype(exer_type)
survey['Smoke'] = survey['Smoke'].astype(smoke_type)

In [ ]:
Verklaring

Standaard ziet pandas de string 'None' als een synoniem voor NaN (ontbrekende data).
Omdat studenten bij de vraag naar sporten letterlijk "Geen" (None) kunnen antwoorden, moeten we keep_default_na=False gebruiken en expliciet meegeven dat alleen de letterlijke tekst 'NA' een ontbrekende waarde is.
Door de kolommen om te zetten naar een geordende CategoricalDtype staan de categorieën straks in een logische volgorde in tabellen en grafieken.

In [ ]:
Opgave

Om voor elke combinatie de effectgrootte (Cramér's V) te berekenen, schrijven we eerst een herbruikbare Python-functie.

Oplossing (Code)

def cramers_v(contingency_table):
    chi2 = stats.chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * min(r - 1, k - 1)))


* Make a frequency table for the two variables. The (presumably) independent variable comes first.
* Plot a graph visualizing the relationship between the two variables.
* Looking at the chart, do you expect a rather high or rather low value for the  χ2  statistic? Why?
* Run the  χ2  test to determine whether there is a relationship between the two variables. Calculate the  χ2  statistic, the critical limit  g  and the  p  value, each for significance level  α=0.05 .
* Should we accept or reject the null hypothesis? What exactly does that mean for the relationship between the two variables? In other words, formulate an answer to the research question.
* Calculate Cramér's V. Do you come to a similar conclusion as with the  χ2  test?

The variables to be investigated:

| Independent variabele          | Dependent variabele                        |
|:------------------------------ |:-------------------------------------------|
| `Exer` (practicing sports)     | `Smoke`                                    |
| `Sex` (gender)                 | `Smoke`                                    |
| `W.Hnd` (dominant hand)        | `Fold` (top hand when you cross your arms) |
| `Sex`                          | `W.Hnd`                                    |

Results of the main calculations (rounded up to 3 decimal places):

- `Exer/Smoke`: χ² ≈ 5.489, g ≈ 12.592, p ≈ 0.483
- `W.Hnd/Fold`: χ² ≈ 1.581, g ≈ 5.992, p ≈ 0.454
- `Sex/Smoke`: χ² ≈ 3.554, g ≈ 7.815, p ≈ 0.314
- `Sex/W.Hnd`: χ² ≈ 0.236, g ≈ 3.842, p ≈ 0.627

Exer/Smoke: χ² ≈ 5.489, g ≈ 12.592, p ≈ 0.483

In [ ]:
3. Analyse van de Variabelenparen
Voor elk paar voeren we telkens dezelfde stappen uit: een kruistabel maken, de χ²-onafhankelijkheidstoets uitvoeren en Cramér's V berekenen. De betrouwbaarheid is telkens α = 0.05.

Paar 1: Exer (Sporten) vs. Smoke (Roken)

# 1. Kruistabel (onafhankelijke variabele eerst = rijen)
exer_smoke_table = pd.crosstab(survey['Exer'], survey['Smoke'])
print("--- Frequentietabel Exer/Smoke ---")
print(exer_smoke_table)

# 2. Chi-kwadraat toets
chi2, p, dof, expected = stats.chi2_contingency(exer_smoke_table)
g = stats.chi2.ppf(1 - 0.05, dof)
v = cramers_v(exer_smoke_table)

print(f"Chi-kwadraat (χ²): {chi2:.3f}")
print(f"Kritieke g-waarde: {g:.3f}")
print(f"p-waarde:          {p:.3f}")
print(f"Cramér's V:        {v:.3f}")

In [ ]:
Verklaring & Conclusie

Resultaat: χ² ≈ 5.489, g ≈ 12.592, p ≈ 0.483, Cramér's V ≈ 0.089

Analyse: De berekende χ²-waarde (5.489) is veel kleiner dan de kritieke g-grens (12.592). Bovendien is de p-waarde (0.483) veel groter dan α = 0.05.

Conclusie: We aanvaarden de nulhypothese (H₀). Er is geen statistisch significant verband tussen hoe vaak een student sport en hoe vaak deze rookt.

W.Hnd/Fold: χ² ≈ 1.581, g ≈ 5.992, p ≈ 0.454

In [ ]:
w_fold_table = pd.crosstab(survey['W.Hnd'], survey['Fold'])
print("\n--- Frequentietabel W.Hnd/Fold ---")
print(w_fold_table)

chi2, p, dof, expected = stats.chi2_contingency(w_fold_table)
g = stats.chi2.ppf(1 - 0.05, dof)
v = cramers_v(w_fold_table)

print(f"Chi-kwadraat (χ²): {chi2:.3f}")
print(f"Kritieke g-waarde: {g:.3f}")
print(f"p-waarde:          {p:.3f}")
print(f"Cramér's V:        {v:.3f}")

In [ ]:
Verklaring & Conclusie

Resultaat: χ² ≈ 1.581, g ≈ 5.992, p ≈ 0.454, Cramér's V ≈ 0.083

Conclusie: We aanvaarden de nulhypothese (H₀). Welke hand dominant is, heeft geen invloed op welke arm bovenop ligt als een student de armen kruist.

Sex/Smoke: χ² ≈ 3.554, g ≈ 7.815, p ≈ 0.314

In [ ]:
sex_smoke_table = pd.crosstab(survey['Sex'], survey['Smoke'])
print("\n--- Frequentietabel Sex/Smoke ---")
print(sex_smoke_table)

chi2, p, dof, expected = stats.chi2_contingency(sex_smoke_table)
g = stats.chi2.ppf(1 - 0.05, dof)
v = cramers_v(sex_smoke_table)

print(f"Chi-kwadraat (χ²): {chi2:.3f}")
print(f"Kritieke g-waarde: {g:.3f}")
print(f"p-waarde:          {p:.3f}")
print(f"Cramér's V:        {v:.3f}")


In [ ]:
Verklaring & Conclusie

Resultaat: χ² ≈ 3.554, g ≈ 7.815, p ≈ 0.314, Cramér's V ≈ 0.123

Conclusie: We aanvaarden de nulhypothese (H₀). Er is geen significant verband tussen het geslacht van de student en het rookgedrag.



Sex/W.Hnd: χ² ≈ 0.236, g ≈ 3.842, p ≈ 0.627

In [ ]:
sex_hand_table = pd.crosstab(survey['Sex'], survey['W.Hnd'])
print("\n--- Frequentietabel Sex/W.Hnd ---")
print(sex_hand_table)

chi2, p, dof, expected = stats.chi2_contingency(sex_hand_table)
g = stats.chi2.ppf(1 - 0.05, dof)
v = cramers_v(sex_hand_table)

print(f"Chi-kwadraat (χ²): {chi2:.3f}")
print(f"Kritieke g-waarde: {g:.3f}")
print(f"p-waarde:          {p:.3f}")
print(f"Cramér's V:        {v:.3f}")

In [ ]:
Verklaring & Conclusie

Resultaat: χ² ≈ 0.236, g ≈ 3.842, p ≈ 0.627, Cramér's V ≈ 0.032

Conclusie: We aanvaarden de nulhypothese (H₀). Of een student man of vrouw is, staat volledig los van het feit of ze links- of rechtshandig zijn.